# ED Agent Mesh — Runnable Pipeline (Day 6)
Clean, consolidated scaffold with identity binding, capacity-aware ICU logic, fallback, scheduling, GRU+MLP adapter, ICU justification, and enricher. Includes two demo scenarios at the end.

In [ ]:

!pip -q install pyyaml
import os, json, time, uuid, hmac, hashlib, yaml
from datetime import datetime, timedelta
from typing import Callable, Dict, Any, List, Tuple, Optional
print("Setup complete.")


In [ ]:

class EventBus:
    def __init__(self): self.subs = {}
    def subscribe(self, topic: str, fn):
        self.subs.setdefault(topic, []).append(fn)
    def publish(self, topic: str, msg: Dict[str, Any]):
        msg = dict(msg); msg['type'] = msg.get('type', topic)
        for pattern, fns in self.subs.items():
            if self._match(topic, pattern):
                for fn in fns: fn({"_topic": topic, **msg})
    @staticmethod
    def _match(topic: str, pattern: str) -> bool:
        if pattern.endswith(".*"): return topic.startswith(pattern[:-2])
        return topic == pattern

BUS = EventBus()
print("Event bus ready.")


In [ ]:

LEDGER_PATH = "/mnt/data/ed_agent_mesh_ledger.jsonl"
LEDGER_INDEX = []

def _hash(data: dict) -> str:
    s = json.dumps(data, sort_keys=True, separators=(",",":"))
    return hashlib.sha256(s.encode()).hexdigest()

def ledger_init():
    global LEDGER_INDEX
    LEDGER_INDEX = []
    os.makedirs(os.path.dirname(LEDGER_PATH), exist_ok=True)
    if os.path.exists(LEDGER_PATH):
        os.remove(LEDGER_PATH)
    genesis = {"kind":"genesis","ts":datetime.utcnow().isoformat(),"prev":None,"payload":{"note":"ED Agent Mesh genesis"}}
    genesis["hash"] = _hash(genesis)
    with open(LEDGER_PATH, "w") as f: f.write(json.dumps(genesis)+"\n")
    LEDGER_INDEX.append(genesis["hash"])

def ledger_append(kind: str, payload: dict) -> dict:
    prev_hash = LEDGER_INDEX[-1] if LEDGER_INDEX else None
    block = {"kind":kind,"ts":datetime.utcnow().isoformat(),"prev":prev_hash,"payload":payload}
    block["hash"] = _hash(block)
    with open(LEDGER_PATH, "a") as f: f.write(json.dumps(block)+"\n")
    LEDGER_INDEX.append(block["hash"]); return block

def ledger_tail(n=20):
    if not os.path.exists(LEDGER_PATH): return []
    with open(LEDGER_PATH) as f: lines = f.readlines()
    return [json.loads(x) for x in lines[-n:]]

ledger_init(); print("Ledger initialized at", LEDGER_PATH)


In [ ]:

SECRET = b"ed-mesh-demo-secret"
def sign(agent: str, payload: dict) -> str:
    raw = json.dumps(payload, sort_keys=True).encode()
    return hmac.new(SECRET, raw, hashlib.sha256).hexdigest()

def base_msg(topic: str, patient: str, encounter: str, payload: dict, meta: dict=None, agent: str="agent"):
    return {
        "type": topic,
        "patient": patient,
        "encounter": encounter,
        "payload": payload or {},
        "meta": meta or {},
        "agent": agent,
        "id": str(uuid.uuid4()),
        "ts": datetime.utcnow().isoformat()
    }


In [ ]:

CAPACITY = {
    "ICU": {"total": 2, "occupied": 1, "jokers": 1},
    "GenWard": {"total": 10, "occupied": 5}
}
def check_capacity(p: dict):
    lvl = (p.get("level") or p.get("payload",{}).get("level") or "").upper()
    if lvl == "ICU":
        free = CAPACITY["ICU"]["total"] - CAPACITY["ICU"]["occupied"]
        return (free > 0), f"ICU free={free}"
    return (True, "OK")
def occupy(unit: str, qty: int=1): CAPACITY[unit]["occupied"] += qty
def release(unit: str, qty: int=1): CAPACITY[unit]["occupied"] = max(0, CAPACITY[unit]["occupied"] - qty)
def joker_available(): return CAPACITY["ICU"].get("jokers",0) > 0
def use_joker():
    if CAPACITY["ICU"]["jokers"]>0:
        CAPACITY["ICU"]["jokers"] -= 1; release("ICU", 1); return True
    return False
print("Capacity helpers ready:", CAPACITY)


In [ ]:

_SIM_T0 = datetime.utcnow(); _SIM_OFFSET = timedelta(0)
def sim_now(): return _SIM_T0 + _SIM_OFFSET
def advance_minutes(n: int):
    global _SIM_OFFSET
    _SIM_OFFSET += timedelta(minutes=n)
    process_pending()

PENDING = {"icu_downgrade": {}, "icu_admit": {}}
ED_ARRIVAL = {}
def mark_ed_arrival(encounter: str): ED_ARRIVAL.setdefault(encounter, sim_now())

def process_pending():
    now = sim_now()
    for enc, task in list(PENDING["icu_downgrade"].items()):
        if task["status"]=="pending" and task["due"] <= now:
            ok = use_joker()
            ledger_append("action", {"kind":"icu_downgrade_complete","encounter":enc,"executed":ok,"ts":now.isoformat()})
            task["status"]="done"
            msg = base_msg("proposal.bed.request", task["patient"], enc, {"service":"InternalMedicine","level":"ICU"}, {}, "fallback")
            BUS.publish("proposal.bed.request", msg)
    for enc, task in list(PENDING["icu_admit"].items()):
        if task["status"]=="pending" and task["due"] <= now:
            ledger_append("action", {"kind":"icu_admit_complete","encounter":enc,"executed":True,"ts":now.isoformat()})
            task["status"]="done"
print("Simulation clock ready.")


In [ ]:

IDENTITY = {}
def identity_bound(encounter: str) -> bool: return encounter in IDENTITY

class IdentityRegistry:
    def __init__(self, bus): bus.subscribe("proposal.identity.bind", self.on_bind)
    def on_bind(self, msg):
        enc = msg.get("ems_id") or msg.get("encounter"); mrn = msg.get("mrn")
        if enc and mrn:
            IDENTITY[enc] = mrn
            ledger_append("action", {"kind":"identity_bound","encounter":enc,"mrn":mrn})
def bind(ems_id: str, mrn: str):
    BUS.publish("proposal.identity.bind", {"ems_id": ems_id, "mrn": mrn, "meta":{"source":"manual"}})
IDENT = IdentityRegistry(BUS)
print("IdentityRegistry online.")


In [ ]:

POLICY = {}
class SafetyGovernor:
    def __init__(self, bus: EventBus, policy: dict):
        self.bus = bus; self.policy = policy
        bus.subscribe("proposal.*", self.evaluate)
    def evaluate(self, msg):
        typ = msg.get("type"); p = (msg.get("payload") or {}); meta = msg.get("meta",{}) or {}; enc = msg.get("encounter")
        if typ in ("proposal.order.labs","proposal.order.ecg","proposal.order.ct","proposal.order.ultrasound","proposal.page"):
            if not identity_bound(enc) and not meta.get("identity_override", False):
                ledger_append("block", {"proposal": msg, "block":{"reason":"Identity not bound EMS↔MRN (set meta.identity_override=True if true emergency)"}})
                BUS.publish("block", {"proposal": msg, "reason":"Identity not bound EMS↔MRN"}); return
        if typ=="proposal.order.ct":
            if (msg.get("payload",{}).get("pregnancy", False)) and not meta.get("attending_override", False):
                ledger_append("block", {"proposal": msg, "block":{"reason":"Pregnancy+ionizing imaging requires attending override"}})
                BUS.publish("block", {"proposal": msg, "reason":"Pregnancy+ionizing imaging requires attending override"}); return
        if typ=="proposal.bed.request":
            level = (p.get("level") or "").upper()
            if level == "ICU":
                ok, why = check_capacity(p)
                if not ok and not meta.get("attending_override", False):
                    ledger_append("block", {"proposal": msg, "block":{"reason":"ICU full — propose ED hold or transfer"}})
                    BUS.publish("block", {"proposal": msg, "reason":"ICU full — propose ED hold or transfer"}); return
        if typ=="proposal.ed_hold":
            ledger_append("permit", {"proposal": msg, "permit":{"reason":"ED hold permitted (ICU full / flow safety)"}})
            BUS.publish("permit.issued", {"proposal": msg, "reason":"ED hold permitted"}); return
        if typ=="proposal.icu_downgrade":
            ledger_append("permit", {"proposal": msg, "permit":{"reason":"ICU downgrade permitted (joker negotiation)"}})
            BUS.publish("permit.issued", {"proposal": msg, "reason":"ICU downgrade permitted"}); return
        ledger_append("permit", {"proposal": msg, "permit":{"reason":"All invariants satisfied"}})
        BUS.publish("permit.issued", {"proposal": msg, "reason":"All invariants satisfied"})
GOVERNOR = SafetyGovernor(BUS, POLICY); print("Governor online.")


In [ ]:

# GRU+MLP + ModelAdapter + PerceptionRisk
import math
try:
    import torch, torch.nn as nn
except Exception as e:
    raise RuntimeError("PyTorch missing; run `!pip install torch --quiet`") from e

FEATURES: List[Tuple[str, float, float, float]] = [
    ("sbp", 50, 250, 120), ("hr", 20, 220, 80), ("rr", 4, 50, 16),
    ("spo2", 50, 100, 97), ("temp_c", 32, 42, 37), ("gcs", 3, 15, 15),
    ("egfr", 5, 150, 90), ("lactate", 0, 10, 1.5), ("trop_hs", 0, 5000, 5),
    ("fio2", 0.21, 1.0, 0.21), ("vasopressor", 0, 1, 0), ("weeks", 0, 42, 0),
    ("pregnancy", 0, 1, 0), ("stemi_ecg", 0, 1, 0), ("fetal_abn", 0, 1, 0),
]
NUM_FEATS = len(FEATURES)
def _clipf(v, lo, hi, d):
    try:
        x = float(v)
        if math.isnan(x) or math.isinf(x): return d
        return max(lo, min(hi, x))
    except Exception:
        return d
def encode_features(feats: Dict[str, Any]):
    xs = []
    for name, lo, hi, d in FEATURES:
        val = feats.get(name, d); val = _clipf(val, lo, hi, d)
        xs.append((val - lo) / (hi - lo) if hi>lo else 0.0)
    import torch; return torch.tensor(xs, dtype=torch.float32).view(1,1,-1)

class HybridGRUMLP(nn.Module):
    def __init__(self, input_dim: int, d_model: int=64, num_layers: int=1, dropout: float=0.1):
        super().__init__()
        self.gru = nn.GRU(input_dim, d_model, num_layers=num_layers, batch_first=True, dropout=0.0 if num_layers==1 else dropout)
        self.body = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Dropout(dropout))
        self.heads = nn.ModuleDict({k: nn.Sequential(nn.Linear(d_model,1), nn.Sigmoid()) for k in [
            "trauma_high","stemi","fetal_concern","icu_need","imc_need","deterioration_6h","boarding_6h"
        ]})
    def forward(self, x):
        h,_ = self.gru(x); z = self.body(h[:,-1,:]); return {k: head(z).squeeze(-1) for k, head in self.heads.items()}

class ModelAdapter:
    def __init__(self, device="cpu", weights_path=None, version="hybrid-gru@wired"):
        import torch
        self.device=device; self.version=version
        self.model = HybridGRUMLP(NUM_FEATS).to(device).eval()
        if weights_path:
            state = torch.load(weights_path, map_location=device); self.model.load_state_dict(state)
        self.thresh = {"stemi":0.80,"trauma_high":0.50,"fetal_concern":0.60,"icu_need":0.60,"imc_need":0.50,"deterioration_6h":0.40,"boarding_6h":0.50}
    def _normalize(self, feats: dict):
        mm = dict(feats)
        for k in ["pregnancy","stemi_ecg","fetal_abn"]:
            if k in mm: mm[k] = 1 if bool(mm[k]) else 0
        return mm
    def predict(self, scenario: str, feats: dict) -> dict:
        import torch
        xdict=self._normalize(feats); xt=encode_features(xdict).to(self.device)
        with torch.no_grad(): out=self.model(xt)
        scores = {k: float(v.squeeze(0).cpu().item()) for k,v in out.items()}
        if scenario=="chest_pain" and xdict.get("stemi_ecg",0)==1:
            scores["stemi"]=max(scores["stemi"],0.95); scores["icu_need"]=max(scores["icu_need"],0.70)
        if scenario=="major_trauma" and (xdict.get("sbp",120)<90 or xdict.get("hr",80)>120):
            scores["trauma_high"]=max(scores["trauma_high"],0.80); scores["icu_need"]=max(scores["icu_need"],0.65)
        return {"model_version": self.version, "scores": scores, "uncertainty": {k:0.2 for k in scores}}

ICU_REQ_SENT = set()
class PerceptionRiskAgent:
    def __init__(self, bus):
        self.bus = bus; self.adapter = ModelAdapter(device="cpu")
    def infer(self, scenario, features):
        pred = self.adapter.predict(scenario, features); s = pred.get("scores", {}); out = {}
        if scenario=="major_trauma": out["trauma_high"] = s.get("trauma_high",0.0) >= self.adapter.thresh["trauma_high"]
        elif scenario=="chest_pain": out["stemi"] = s.get("stemi",0.0) >= self.adapter.thresh["stemi"]
        elif scenario=="pregnant_syncope":
            out["late_trimester"] = (features.get("weeks",0)>=28)
            out["fetal_concern"]  = s.get("fetal_concern",0.0) >= self.adapter.thresh["fetal_concern"]
        out.update({
            "icu_need": s.get("icu_need",0.0) >= self.adapter.thresh["icu_need"],
            "imc_need": s.get("imc_need",0.0) >= self.adapter.thresh["imc_need"],
            "deterioration_6h": s.get("deterioration_6h",0.0) >= self.adapter.thresh["deterioration_6h"],
            "boarding_6h": s.get("boarding_6h",0.0) >= self.adapter.thresh["boarding_6h"],
            "_raw": s, "_model_version": pred.get("model_version")
        })
        return out
    def publish(self, patient, encounter, scenario, features):
        mark_ed_arrival(encounter)
        scores = self.infer(scenario, features)
        hyp = base_msg("hypothesis."+scenario, patient, encounter, {"scores": scores, "features": features}, agent="perception.risk")
        hyp["sig"] = sign("perception.risk", hyp); BUS.publish("hypothesis."+scenario, hyp)
        try:
            icu_bool = bool(scores.get("icu_need", False)); icu_raw = float((scores.get("_raw") or {}).get("icu_need", 0.0))
        except Exception: icu_bool, icu_raw = False, 0.0
        if icu_bool and encounter not in ICU_REQ_SENT:
            ICU_REQ_SENT.add(encounter)
            meta = {"justification_text": f"High ICU probability ({icu_raw:.2f}) — early bed request",
                    "model_version": scores.get("_model_version"), "icu_prob": icu_raw}
            req = base_msg("proposal.bed.request", patient, encounter, {"service":"InternalMedicine","level":"ICU"}, meta, agent="perception.risk")
            req["sig"] = sign("perception.risk", req); BUS.publish("proposal.bed.request", req)
print("PerceptionRisk wired to GRU+MLP.")


In [ ]:

class CriticAgent:
    def __init__(self, bus): self.bus = bus; bus.subscribe("hypothesis.*", self.on_hyp)
    def on_hyp(self, msg): ledger_append("audit", {"kind":"hypothesis", "encounter": msg.get("encounter")})

class DiagnosticsCoordinator:
    def __init__(self, bus): self.bus = bus; bus.subscribe("hypothesis.*", self.on_hyp)
    def on_hyp(self, msg):
        enc = msg["encounter"]; patient = msg["patient"]
        BUS.publish("proposal.order.ecg", base_msg("proposal.order.ecg", patient, enc, {}, {}, "diag"))
        BUS.publish("proposal.order.labs", base_msg("proposal.order.labs", patient, enc, {}, {}, "diag"))
        scores = msg.get("payload",{}).get("scores",{})
        if scores.get("trauma_high") or scores.get("deterioration_6h"):
            BUS.publish("proposal.order.ct", base_msg("proposal.order.ct", patient, enc, {}, {}, "diag"))

class CapacityCoordinator:
    def __init__(self, bus): self.bus = bus; bus.subscribe("hypothesis.*", self.on_hyp)
    def on_hyp(self, msg):
        scores = msg.get("payload",{}).get("scores",{})
        if scores.get("stemi"):
            BUS.publish("proposal.page", base_msg("proposal.page", msg["patient"], msg["encounter"], {"role":"Cath lab"}, {}, "capacity"))
        if scores.get("trauma_high") or scores.get("icu_need"):
            BUS.publish("proposal.bed.request", base_msg("proposal.bed.request", msg["patient"], msg["encounter"], {"service":"InternalMedicine","level":"ICU"}, {}, "capacity"))

FALLBACK_STATE = {}; FALLBACK_COOLDOWN_MIN = 15
class FallbackAgent:
    def __init__(self, bus):
        self.bus = bus; bus.subscribe("block.issued", self.on_block); bus.subscribe("block", self.on_block)
    def _should_emit(self, enc: str, reason: str):
        now = sim_now(); st = FALLBACK_STATE.get(enc, {}); last_r = st.get("last_reason"); last_t = st.get("last_ts")
        return (last_r != reason) or (last_t is None) or ((now - last_t) >= timedelta(minutes=FALLBACK_COOLDOWN_MIN))
    def on_block(self, msg):
        prop = msg.get("proposal") or {}; typ = prop.get("type"); reason = msg.get("reason",""); patient = prop.get("patient"); enc = prop.get("encounter")
        if typ != "proposal.bed.request" or "ICU full" not in reason: return
        if not self._should_emit(enc, reason): return
        FALLBACK_STATE[enc] = {"last_reason": reason, "last_ts": sim_now(), "ed_hold_sent": FALLBACK_STATE.get(enc,{}).get("ed_hold_sent", False)}
        if not FALLBACK_STATE[enc]["ed_hold_sent"]:
            BUS.publish("proposal.ed_hold", base_msg("proposal.ed_hold", patient, enc, {"reason":"ICU full","service":"ED"}, {}, "fallback"))
            FALLBACK_STATE[enc]["ed_hold_sent"] = True
        if joker_available():
            BUS.publish("proposal.icu_downgrade", base_msg("proposal.icu_downgrade", patient, enc, {"target_ward":"GenWard"}, {}, "fallback"))
        else:
            BUS.publish("proposal.transfer.query", base_msg("proposal.transfer.query", patient, enc, {"level":"ICU","region":"metro"}, {}, "fallback"))

ICU_MOVE_MIN = 120; ICU_ADMIT_MIN = 120
class ActuatorAgent:
    def __init__(self, bus: EventBus):
        self.bus = bus; bus.subscribe("permit.issued", self.on_permit)
        for t in ["proposal.order.labs","proposal.order.ecg","proposal.page","proposal.order.ultrasound","proposal.ed_hold","proposal.transfer.query"]:
            bus.subscribe(t, self.on_low_risk)
    def _identity_ok(self, msg):
        enc = msg.get("encounter"); meta = msg.get("meta",{}) or {}
        if msg.get("type")=="proposal.ed_hold": return True
        return identity_bound(enc) or meta.get("identity_override", False)
    def on_low_risk(self, msg):
        if not self._identity_ok(msg):
            ledger_append("prevented", {"reason":"Identity not bound; low-risk auto-exec suppressed","proposal": msg}); return
        ledger_append("action", {"proposal": msg, "executed": True})
    def on_permit(self, msg):
        prop = msg.get("proposal", {}); typ = prop.get("type",""); enc = prop.get("encounter"); pat = prop.get("patient"); now = sim_now()
        if typ=="proposal.icu_downgrade":
            due = now + timedelta(minutes=ICU_MOVE_MIN)
            PENDING["icu_downgrade"][enc] = {"due":due,"status":"pending","patient":pat}
            ledger_append("action", {"kind":"icu_downgrade_scheduled","encounter":enc,"due":due.isoformat()}); return
        if typ=="proposal.bed.request":
            lvl = (prop.get("payload",{}).get("level") or "Ward").upper()
            if lvl=="ICU":
                due = now + timedelta(minutes=ICU_ADMIT_MIN)
                PENDING["icu_admit"][enc] = {"due":due,"status":"pending","patient":pat,"created":now}
                ledger_append("action", {"kind":"icu_admit_scheduled","encounter":enc,"due":due.isoformat()}); return
        ledger_append("action", {"permit": msg, "executed": True})

def _news2_stub(feats):
    rr = feats.get("rr",16); spo2 = feats.get("spo2",97); sbp = feats.get("sbp",120); hr = feats.get("hr",80); temp = feats.get("temp_c",37); gcs = feats.get("gcs",15)
    score = 0
    score += 3 if rr>=25 else (2 if rr>=21 else (1 if rr>=12 and rr<=20 else 0))
    score += 3 if spo2<91 else (2 if spo2<93 else (1 if spo2<95 else 0))
    score += 3 if sbp<=90 else (2 if sbp<=100 else 0)
    score += 3 if hr>=130 else (2 if hr>=111 else (1 if hr>=91 else 0))
    score += 1 if temp<36 or temp>=39 else 0
    score += 3 if gcs<15 else 0
    return float(score)

class ICUJustificationAgent:
    def __init__(self, bus, icu_thresh=0.6):
        self.bus = bus; self.thresh = icu_thresh; bus.subscribe("hypothesis.*", self.on_hypothesis)
    def on_hypothesis(self, msg):
        p = msg.get("payload",{}) or {}; scores = p.get("scores",{}) or {}; feats = p.get("features",{}) or {}
        icu_bool = bool(scores.get("icu_need", False)); raw = scores.get("_raw", {}) if isinstance(scores, dict) else {}
        icu_raw = float(raw.get("icu_need", 0.0)) if isinstance(raw, dict) else 0.0
        if not (icu_bool or icu_raw >= self.thresh): return
        just = {
            "NEWS2": float(feats.get("news2", _news2_stub(feats))), "SBP": feats.get("sbp"), "HR": feats.get("hr"), "RR": feats.get("rr"),
            "SpO2": feats.get("spo2"), "FiO2": feats.get("fio2"), "Lactate": feats.get("lactate"),
            "GCS": feats.get("gcs"), "Vasopressor": feats.get("vasopressor",0),
            "ModelScores": {k: float(v) for k,v in (raw or {}).items()}, "ModelVersion": scores.get("_model_version")
        }
        m = base_msg("proposal.bed.request", msg["patient"], msg["encounter"], {"service":"InternalMedicine","level":"ICU"}, {"justification":just}, "coord.capacity")
        m["sig"] = sign("coord.capacity", m); self.bus.publish("proposal.bed.request", m)

class ProposalEnricher:
    def __init__(self, bus):
        self.bus = bus; bus.subscribe("proposal.bed.request", self.on_bed_request); self.cache = {}
        bus.subscribe("hypothesis.*", self.cache_feats)
    def cache_feats(self, msg): self.cache[msg["encounter"]] = msg.get("payload",{})
    def on_bed_request(self, msg):
        meta = msg.setdefault("meta", {})
        if "justification" not in meta:
            feats = (self.cache.get(msg.get("encounter"),{}) or {}).get("features",{})
            raw = (self.cache.get(msg.get("encounter"),{}) or {}).get("scores",{})
            meta["justification"] = {
                "NEWS2": float(feats.get("news2", _news2_stub(feats))), "SBP": feats.get("sbp"), "HR": feats.get("hr"), "RR": feats.get("rr"),
                "SpO2": feats.get("spo2"), "FiO2": feats.get("fio2"), "Lactate": feats.get("lactate"), "GCS": feats.get("gcs"),
                "Vasopressor": feats.get("vasopressor",0), "ModelScores": {k: float(v) for k,v in (raw.get('_raw',{}) if isinstance(raw,dict) else {}).items()},
                "ModelVersion": raw.get("_model_version") if isinstance(raw,dict) else None
            }
            msg["meta"] = meta

class AuditAgent:
    def __init__(self, bus):
        self.bus = bus
        for t in ["block","permit.issued","permit","action"]: bus.subscribe(t, self.on_evt)
    def on_evt(self, msg):
        k = "audit" if msg.get("_topic") not in ("block","permit","permit.issued","action") else msg.get("_topic")
        if k=="block": ledger_append("block", {"proposal": msg.get("proposal",{}), "block":{"reason": msg.get("reason","")}})
        elif k in ("permit","permit.issued"): ledger_append("audit", {"kind":"permit","reason": msg.get("reason","")})
        elif k=="action": ledger_append("audit", {"kind":"action"})


In [ ]:

ledger_init()
BUS = EventBus()
GOVERNOR = SafetyGovernor(BUS, {})
IDENT = IdentityRegistry(BUS)
RISK = PerceptionRiskAgent(BUS)
CRITIC = CriticAgent(BUS)
DIAG = DiagnosticsCoordinator(BUS)
CAP = CapacityCoordinator(BUS)
ACT = ActuatorAgent(BUS)
FALLBACK = FallbackAgent(BUS)
ICUJUST = ICUJustificationAgent(BUS, icu_thresh=0.6)
ENRICH = ProposalEnricher(BUS)
AUD = AuditAgent(BUS)
print("Mesh online. ICU capacity:", CAPACITY["ICU"])


## Demo A — ICU available (straight permit + scheduled admit)

In [ ]:

ledger_init()
CAPACITY["ICU"]["occupied"] = max(0, CAPACITY["ICU"]["total"] - 1)
enc, pat = "ems-demoA", "tmp-demoA"
bind(enc, "MRN-A")
RISK.publish(pat, enc, "major_trauma", {"sbp":95,"hr":122,"rr":26,"spo2":90,"gcs":13,"lactate":3.2,"fio2":0.4})
time.sleep(0.2)
print("— after proposals —")
for r in [json.loads(x) for x in open(LEDGER_PATH)]:
    k=r["kind"]; pay=r.get("payload",{}); prop=pay.get("proposal",{})
    if prop.get("encounter")==enc and k in ("block","permit","action","audit"):
        reason=(pay.get("block") or pay.get("permit") or {}).get("reason",""); note=pay.get("note","") or pay.get("kind","")
        print(k, prop.get("type","") if prop else "", "|", reason or note)
advance_minutes(120)
print("— after +2h —")
for r in [json.loads(x) for x in open(LEDGER_PATH)][-20:]:
    k=r["kind"]; pay=r.get("payload",{}); prop=pay.get("proposal",{})
    if k in ("block","permit","action","audit"):
        reason=(pay.get("block") or pay.get("permit") or {}).get("reason",""); print(k, prop.get("type","") if prop else "", "|", reason or pay.get("note",""))


## Demo B — ICU full (block → ED hold → joker → re-request → admit)

In [ ]:

ledger_init()
CAPACITY["ICU"]["occupied"] = CAPACITY["ICU"]["total"]
CAPACITY["ICU"]["jokers"] = 1
enc, pat = "ems-demoB", "tmp-demoB"
bind(enc, "MRN-B")
RISK.publish(pat, enc, "major_trauma", {"sbp":95,"hr":122,"rr":26,"spo2":90,"gcs":13,"lactate":3.2,"fio2":0.4})
time.sleep(0.2)
print("— after arrival —")
for r in [json.loads(x) for x in open(LEDGER_PATH)]:
    k=r["kind"]; pay=r.get("payload",{}); prop=pay.get("proposal",{})
    if prop.get("encounter")==enc and k in ("block","permit","action","audit"):
        reason=(pay.get("block") or pay.get("permit") or {}).get("reason",""); note=pay.get("note","") or pay.get("kind","")
        print(k, prop.get("type","") if prop else "", "|", reason or note)

advance_minutes(120)
print("— after +2h —")
for r in [json.loads(x) for x in open(LEDGER_PATH)][-30:]:
    k=r["kind"]; pay=r.get("payload",{}); prop=pay.get("proposal",{})
    if k in ("block","permit","action","audit"):
        reason=(pay.get("block") or pay.get("permit") or {}).get("reason",""); print(k, prop.get("type","") if prop else "", "|", reason or pay.get("note",""))

advance_minutes(120)
print("— after +4h —")
for r in [json.loads(x) for x in open(LEDGER_PATH)][-30:]:
    k=r["kind"]; pay=r.get("payload",{}); prop=pay.get("proposal",{})
    if k in ("block","permit","action","audit"):
        reason=(pay.get("block") or pay.get("permit") or {}).get("reason",""); print(k, prop.get("type","") if prop else "", "|", reason or pay.get("note",""))
